# Idealista Web Scraper — Local, No-AWS Guided Walkthrough

**FEATURE-002, task 1.11.** This notebook is a guided, cell-by-cell surface for developing and
testing the Idealista web-scraper OOP core (`data_collection.scraper`) **entirely locally**: no AWS
credentials, no paid proxy vendor. It wires `NullProxyProvider` + `LocalListingRepository` so every
cell can be run by anyone who clones this repo.

**Compliance / ToS note (REVIEW-FEATURE-002 finding H2):** this scraper targets Idealista's public
search-results pages to supplement the official API collector (capped at 100 listings/month). Please
scrape responsibly:
- keep the randomised inter-page delay (`ScrapeOrchestrator`'s default 2.0–4.5s) — never tighten it,
- respect the `SCRAPER_ENABLED` kill switch (`config.py`) — set it to `false` to disable any run,
- prefer the official API collector (`bronze_collector.py`) as the primary source where its quota
  suffices; this scraper is a supplement, not a replacement,
- this notebook makes **at most a handful of requests** per run (1 page for the guided fetch, 2–3
  pages per operation for the smoke-test run below) — it is not intended to scrape the full inventory
  interactively.


## simple test

In [ ]:
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
import time
import random

# https://stackoverflow.com/questions/70485179/runtimeerror-when-using-undetected-chromedriver
# version_main must match the installed Chrome major version (currently 152).
driver = uc.Chrome(use_subprocess=True, version_main=152)

time.sleep(random.uniform(2, 5))  # Random sleep to mimic human behavior  
property_url = "https://www.idealista.com/inmueble/106749418/"
driver.get(property_url)

In [10]:
from dataclasses import dataclass, field
from typing import Optional

@dataclass
class PropertyListing:
    # Identifikation
    url: str
    idealista_id: str
    
    # Preise
    price_eur: Optional[int] = None
    prev_price_eur: Optional[int] = None
    price_drop_pct: Optional[float] = None
    price_per_sqm: Optional[float] = None
    community_fee_eur_month: Optional[int] = None
    
    # Fläche
    sqm_built: Optional[int] = None        # construidos
    sqm_usable: Optional[int] = None       # útiles
    
    # Ausstattung
    rooms: Optional[int] = None
    bathrooms: Optional[int] = None
    floor: Optional[str] = None
    is_exterior: Optional[bool] = None     # True=exterior, False=interior
    orientation: list[str] = field(default_factory=list)
    has_elevator: bool = False
    has_balcony: bool = False
    has_parking: bool = False
    has_ac: bool = False
    has_fitted_wardrobes: bool = False
    has_heating: Optional[bool] = None
    heating_type: Optional[str] = None     # e.g. "natural gas"
    
    # Lage & Meta
    title: str = ""
    location: str = ""
    description: str = ""                   # "Comentario del anunciante"
    neighborhood: Optional[str] = None
    district: Optional[str] = None
    city: Optional[str] = None
    latitude: Optional[float] = None
    longitude: Optional[float] = None
    year_built: Optional[int] = None
    condition: Optional[str] = None        # z.B. "good", "new", "needs_renovation"
    
    # Rohdaten als Fallback
    raw_features: list[str] = field(default_factory=list)

In [11]:
import re
from dataclasses import asdict
from urllib.parse import unquote


def _extract_idealista_id_from_url(url: str) -> str:
    """Extract the listing id from URLs like .../inmueble/111622319/."""
    match = re.search(r"/inmueble/(\d+)/", url)
    return match.group(1) if match else ""


def _parse_int_from_text(value: str | None) -> int | None:
    """Parse first integer from text, handling thousands separators."""
    if not value:
        return None
    digits = re.sub(r"[^\d]", "", value)
    return int(digits) if digits else None


def _parse_pct_from_text(value: str | None) -> float | None:
    """Parse percentages like '-8%' or '8,5 %' into float."""
    if not value:
        return None
    cleaned = value.replace(",", ".")
    match = re.search(r"-?\d+(?:\.\d+)?", cleaned)
    return float(match.group(0)) if match else None


def _extract_community_fee_eur_month(driver) -> int | None:
    """Read 'Gastos de comunidad' from the price box (e.g. '45 €/mes')."""
    rows = driver.find_elements(
        By.XPATH,
        "//section[contains(@class, 'price-features__container')]//p[contains(@class, 'flex-feature')]",
    )
    for row in rows:
        label_elements = row.find_elements(By.XPATH, ".//span[contains(@class, 'flex-feature-text')]")
        detail_elements = row.find_elements(By.XPATH, ".//span[contains(@class, 'flex-feature-details')]")
        if not label_elements or not detail_elements:
            continue
        label = label_elements[0].text.lower().strip()
        if "gastos de comunidad" in label:
            return _parse_int_from_text(detail_elements[0].text)
    return None


def _extract_location_parts(driver) -> tuple[Optional[str], Optional[str], Optional[str]]:
    """Parse neighborhood, district and city from the Ubicación block."""
    rows = driver.find_elements(By.XPATH, "//div[@id='headerMap']//li[contains(@class, 'header-map-list')]")
    texts = [row.text.strip() for row in rows if row.text.strip()]

    neighborhood: Optional[str] = None
    district: Optional[str] = None
    city: Optional[str] = None

    for text in texts:
        lower_text = text.lower()
        if lower_text.startswith("barrio "):
            neighborhood = text.split(" ", 1)[1].strip()
        elif lower_text.startswith("distrito "):
            district = text.split(" ", 1)[1].strip()
        elif lower_text.startswith("valencia"):
            city = "Valencia"

    # Fallback for pages that do not expose the exact same Ubicación structure.
    if city is None and texts:
        city = texts[-1].split(",")[0].strip()

    return neighborhood, district, city


def _parse_coords_from_text(text: str) -> tuple[Optional[float], Optional[float]]:
    """Parse lat/lon from URL or script snippets containing center/markers."""
    center_match = re.search(r"[?&]center=(-?\d+(?:\.\d+)?),(-?\d+(?:\.\d+)?)", text)
    if center_match:
        return float(center_match.group(1)), float(center_match.group(2))

    marker_match = re.search(r"markers=[^&]*?(-?\d+(?:\.\d+)?),(-?\d+(?:\.\d+)?)", text)
    if marker_match:
        return float(marker_match.group(1)), float(marker_match.group(2))

    latlng_match = re.search(r"LatLng\((-?\d+(?:\.\d+)?),\s*(-?\d+(?:\.\d+)?)\)", text)
    if latlng_match:
        return float(latlng_match.group(1)), float(latlng_match.group(2))

    return None, None


def _extract_coordinates(driver) -> tuple[Optional[float], Optional[float]]:
    """Extract approximate coordinates from static map URLs (center/markers)."""
    url_candidates: list[str] = []

    map_images = driver.find_elements(By.XPATH, "//div[@id='map']//img[contains(@src, 'staticmap')]")
    map_images += driver.find_elements(By.XPATH, "//img[@id='sMap']")
    for img in map_images:
        src = img.get_attribute("src") or ""
        if src:
            url_candidates.append(src)

    show_map_links = driver.find_elements(By.XPATH, "//a[contains(@class, 'showMap')]")
    for link in show_map_links:
        href = link.get_attribute("href") or ""
        if href:
            url_candidates.append(href)

    for candidate in url_candidates:
        lat, lon = _parse_coords_from_text(unquote(candidate))
        if lat is not None and lon is not None:
            return lat, lon

    # Last fallback: search the full page source for center/markers/LatLng snippets.
    page_source = driver.page_source or ""
    lat, lon = _parse_coords_from_text(unquote(page_source))
    if lat is not None and lon is not None:
        return lat, lon

    return None, None


def _expand_description(driver) -> None:
    """Click the 'Ver más' toggle so the full (collapsed) description is rendered."""
    toggles = driver.find_elements(
        By.XPATH,
        "//div[contains(@class, 'adCommentsLanguage')]"
        "//*[contains(@class, 'expandable') or contains(@class, 'see-more') "
        "or contains(@class, 'link-underline') or contains(text(), 'Ver más') "
        "or contains(text(), 'más')]",
    )
    for toggle in toggles:
        try:
            # JS click avoids interception by overlays/cookie banners.
            driver.execute_script("arguments[0].click();", toggle)
            time.sleep(random.uniform(0.3, 0.8))
        except Exception:
            continue


def _extract_description(driver) -> str:
    """Read the advertiser's free-text description ('Comentario del anunciante')."""
    _expand_description(driver)
    # The description lives in the Spanish comment block; other language tabs are
    # hidden translation duplicates, so target adCommentsLanguage explicitly.
    comment_elements = driver.find_elements(
        By.XPATH,
        "//div[contains(@class, 'adCommentsLanguage')]//p",
    )
    parts = [element.text.strip() for element in comment_elements if element.text.strip()]
    return "\n\n".join(parts)


def _parse_orientations(feature_text: str) -> list[str]:
    """Extract orientation directions (norte/sur/este/oeste, etc.) from text."""
    directions = [
        "norte",
        "sur",
        "este",
        "oeste",
        "noreste",
        "noroeste",
        "sureste",
        "suroeste",
    ]
    found: list[str] = []
    for direction in directions:
        if re.search(rf"\b{direction}\b", feature_text):
            found.append(direction)
    return found


def parse_features(raw_features: list[str]) -> dict:
    """Parse raw Idealista feature strings into structured fields."""
    result = {
        "sqm_built": None,
        "sqm_usable": None,
        "rooms": None,
        "bathrooms": None,
        "floor": None,
        "is_exterior": None,
        "orientation": [],
        "year_built": None,
        "condition": None,
        "has_balcony": False,
        "has_elevator": False,
        "has_ac": False,
        "has_fitted_wardrobes": False,
        "has_parking": False,
        "has_heating": None,
        "heating_type": None,
    }

    for feature in raw_features:
        f = feature.lower().strip()

        m = re.search(r"(\d+)\s*m²\s*construidos", f)
        if m:
            result["sqm_built"] = int(m.group(1))

        m = re.search(r"(\d+)\s*m²\s*[uú]tiles", f)
        if m:
            result["sqm_usable"] = int(m.group(1))

        m = re.search(r"(\d+)\s*habitacion", f)
        if m:
            result["rooms"] = int(m.group(1))

        m = re.search(r"(\d+)\s*ba[ñn]", f)
        if m:
            result["bathrooms"] = int(m.group(1))

        m = re.search(r"(\d+)[aª°]?\s*planta", f)
        if m:
            result["floor"] = m.group(1)

        # Exterior/interior may appear in the same or a separate feature row.
        if "exterior" in f:
            result["is_exterior"] = True
        elif "interior" in f and result["is_exterior"] is None:
            result["is_exterior"] = False

        m = re.search(r"construido en\s*(\d{4})", f)
        if m:
            result["year_built"] = int(m.group(1))

        # Keep para reformar separate from good/new condition.
        if "para reformar" in f:
            result["condition"] = "needs_renovation"
        elif "buen estado" in f or "segunda mano" in f:
            result["condition"] = "good"
        elif "obra nueva" in f or "nuevo" in f:
            result["condition"] = "new"

        if "balc" in f:
            result["has_balcony"] = True
        if "ascensor" in f:
            result["has_elevator"] = True
        if "aire acondicionado" in f:
            result["has_ac"] = True
        if "armarios empotrados" in f:
            result["has_fitted_wardrobes"] = True
        if "garaje" in f or "parking" in f:
            result["has_parking"] = True

        # Heating detection with explicit negative handling.
        if "no dispone de calefacci" in f or "sin calefacci" in f:
            result["has_heating"] = False
            result["heating_type"] = None
        elif "calefacci" in f:
            if result["has_heating"] is None:
                result["has_heating"] = True
            if "gas natural" in f:
                result["heating_type"] = "natural gas"

        if "orientaci" in f:
            parsed_orientations = _parse_orientations(f)
            if parsed_orientations:
                result["orientation"] = parsed_orientations

    return result


def extract_property_details(driver, property_url: str) -> PropertyListing:
    """Extract and map listing details into a strongly-typed PropertyListing dataclass."""
    title = driver.find_element(By.XPATH, './/span[contains(@class, "main-info__title-main")]').text
    description = _extract_description(driver)  
    location = driver.find_element(By.XPATH, '//span[contains(@class, "main-info__title-minor")]').text
    neighborhood, district, city = _extract_location_parts(driver)
    latitude, longitude = _extract_coordinates(driver)
    price_text = driver.find_element(By.XPATH, '//span[contains(@class, "info-data-price")]').text

    prev_price_elements = driver.find_elements(By.XPATH, '//span[contains(@class, "pricedown_price")]')
    prev_price_text = prev_price_elements[0].text.strip() if prev_price_elements else None

    price_drop_elements = driver.find_elements(By.XPATH, '//span[contains(@class, "pricedown_icon")]')
    price_drop_text = price_drop_elements[0].text.strip() if price_drop_elements else None

    feature_items = driver.find_elements(By.XPATH, "//div[contains(@class, 'details-property_features')]//li")
    raw_features = [li.text.strip() for li in feature_items if li.text.strip()]
    features = parse_features(raw_features)

    price_eur = _parse_int_from_text(price_text)
    sqm_built = features["sqm_built"]
    price_per_sqm = (price_eur / sqm_built) if price_eur and sqm_built else None

    listing = PropertyListing(
        url=property_url,
        idealista_id=_extract_idealista_id_from_url(property_url),
        price_eur=price_eur,
        prev_price_eur=_parse_int_from_text(prev_price_text),
        price_drop_pct=_parse_pct_from_text(price_drop_text),
        price_per_sqm=price_per_sqm,
        community_fee_eur_month=_extract_community_fee_eur_month(driver),
        sqm_built=features["sqm_built"],
        sqm_usable=features["sqm_usable"],
        rooms=features["rooms"],
        bathrooms=features["bathrooms"],
        floor=features["floor"],
        is_exterior=features["is_exterior"],
        orientation=features["orientation"],
        has_elevator=features["has_elevator"],
        has_balcony=features["has_balcony"],
        has_parking=features["has_parking"],
        has_ac=features["has_ac"],
        has_fitted_wardrobes=features["has_fitted_wardrobes"],
        has_heating=features["has_heating"],
        heating_type=features["heating_type"],
        title=title,
        description=description,
        location=location,
        neighborhood=neighborhood,
        district=district,
        city=city,
        latitude=latitude,
        longitude=longitude,
        year_built=features["year_built"],
        condition=features["condition"],
        raw_features=raw_features,
    )
    return listing

listing = extract_property_details(driver, property_url)

asdict(listing)


{'url': 'https://www.idealista.com/inmueble/106749418/',
 'idealista_id': '106749418',
 'price_eur': 1350000,
 'prev_price_eur': None,
 'price_drop_pct': None,
 'price_per_sqm': 4500.0,
 'community_fee_eur_month': None,
 'sqm_built': 300,
 'sqm_usable': None,
 'rooms': 3,
 'bathrooms': 5,
 'floor': '6',
 'is_exterior': True,
 'orientation': ['norte', 'oeste'],
 'has_elevator': True,
 'has_balcony': True,
 'has_parking': True,
 'has_ac': True,
 'has_fitted_wardrobes': True,
 'has_heating': True,
 'heating_type': 'natural gas',
 'title': 'Piso en venta en Exposició',
 'location': 'El Pla del Real, València',
 'description': 'Descubre este espectacular piso de 300 m² en la prestigiosa zona de Alameda, donde el lujo y la comodidad se fusionan en un entorno privilegiado. Su ubicación es inmejorable, a un paso del cauce del río Turia y a pocos minutos del vibrante centro de la ciudad. Disfruta de la cercanía a todos los servicios esenciales como supermercados, colegios y hospitales, además d

In [12]:
# Zugriff auf einzelne Felder der Dataclass
print(listing.idealista_id)
print(listing.price_eur)
print(listing.neighborhood)
print(listing.district)
print(listing.city)
print(listing.latitude)
print(listing.longitude)
print(listing.community_fee_eur_month)
print(listing.condition)
print(listing.is_exterior)
print(listing.orientation)
print(listing.has_heating)
print(listing.heating_type)
print(listing.has_ac)
print(listing.has_fitted_wardrobes)

# Falls du später JSON schreiben willst:
from dataclasses import asdict
asdict(listing)

106749418
1350000
Exposició
El Pla del Real
Valencia
39.4766666
-0.361475
None
good
True
['norte', 'oeste']
True
natural gas
True
True


{'url': 'https://www.idealista.com/inmueble/106749418/',
 'idealista_id': '106749418',
 'price_eur': 1350000,
 'prev_price_eur': None,
 'price_drop_pct': None,
 'price_per_sqm': 4500.0,
 'community_fee_eur_month': None,
 'sqm_built': 300,
 'sqm_usable': None,
 'rooms': 3,
 'bathrooms': 5,
 'floor': '6',
 'is_exterior': True,
 'orientation': ['norte', 'oeste'],
 'has_elevator': True,
 'has_balcony': True,
 'has_parking': True,
 'has_ac': True,
 'has_fitted_wardrobes': True,
 'has_heating': True,
 'heating_type': 'natural gas',
 'title': 'Piso en venta en Exposició',
 'location': 'El Pla del Real, València',
 'description': 'Descubre este espectacular piso de 300 m² en la prestigiosa zona de Alameda, donde el lujo y la comodidad se fusionan en un entorno privilegiado. Su ubicación es inmejorable, a un paso del cauce del río Turia y a pocos minutos del vibrante centro de la ciudad. Disfruta de la cercanía a todos los servicios esenciales como supermercados, colegios y hospitales, además d